# Tiny Dreamer Highway — Screening Run

**Name:** Esteban  
**Course:** CSC 580 AI 2  
**Assignment:** Final Project — Dream the Road  
**AI tools consulted:** GitHub Copilot

This notebook is designed to reduce wasted H100 time. Instead of launching one long experiment and hoping it works, it runs a **staged screening schedule** with:

- a **safer config** than the aggressive AMP profile
- **short checkpoints** at multiple cycle counts
- a **quick real-environment demo** after each stage
- optional **auto-stop** if the policy still looks bad

## Strategy

The previous H100 AMP run was likely too aggressive: large batch, long imagination horizon, and too many behavior updates let the actor exploit model errors. This notebook uses a more conservative setting and checks **real-environment reward during training** at staged checkpoints before committing to a long run.

The key idea is to run cheap periodic evaluation rollouts on the real environment every few checkpoints, rather than waiting until the very end of a long experiment.

In [2]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/CSC_580_Final_Project')
ARTIFACT_ROOT = DRIVE_ROOT / 'artifacts'

for path in [DRIVE_ROOT, ARTIFACT_ROOT, ARTIFACT_ROOT / 'training_runs']:
    path.mkdir(parents=True, exist_ok=True)

print('Drive root:', DRIVE_ROOT)

Mounted at /content/drive
Drive root: /content/drive/MyDrive/CSC_580_Final_Project


In [3]:
%%bash
set -e
REPO_URL='https://github.com/estmon8u/CSC_580_Final_Project.git'
if [ ! -d /content/CSC_580_Final_Project/.git ]; then
  git clone "${REPO_URL}" /content/CSC_580_Final_Project
else
  cd /content/CSC_580_Final_Project
  git pull --ff-only origin main
fi
cd /content/CSC_580_Final_Project
python -m pip install --upgrade pip --quiet
python -m pip install -e . --quiet
python -m pip install flashoptim --quiet || echo "flashoptim unavailable, using standard AdamW"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 36.2 MB/s eta 0:00:00


Cloning into '/content/CSC_580_Final_Project'...


In [4]:
import json
import sys
from pprint import pprint

import torch
import yaml

PROJECT_ROOT = Path('/content/CSC_580_Final_Project')
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from tiny_dreamer_highway.config import load_experiment_config
from tiny_dreamer_highway.evaluation import record_demo_videos
from tiny_dreamer_highway.training import run_training_experiment

CONFIG_PATH = PROJECT_ROOT / 'examples' / 'h100_screening_experiment.yaml'
config = load_experiment_config(CONFIG_PATH)
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'

print('Loaded config from:', CONFIG_PATH)
print('GPU:', gpu_name)
if 'H100' not in gpu_name:
    print('Warning: this notebook is intended for H100-class runtimes.')

Loaded config from: /content/CSC_580_Final_Project/examples/h100_screening_experiment.yaml
GPU: NVIDIA H100 80GB HBM3


In [5]:
RUN_NAME = 'h100_screen_run_001'
RUN_ARTIFACT_ROOT = ARTIFACT_ROOT / 'training_runs' / RUN_NAME
RUN_ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

config_snapshot_path = RUN_ARTIFACT_ROOT / 'screening_config.yaml'
config_snapshot_path.write_text(
    yaml.safe_dump(config.model_dump(mode='python'), sort_keys=False),
    encoding='utf-8',
)

print('Run name:', RUN_NAME)
print('Saved config snapshot to:', config_snapshot_path)
pprint({
    'max_episode_steps': config.env.max_episode_steps,
    'batch_size': config.training.batch_size,
    'imagination_horizon': config.training.imagination_horizon,
    'world_model_lr': config.training.world_model_lr,
    'actor_lr': config.training.actor_lr,
    'critic_lr': config.training.critic_lr,
    'world_model_updates_per_cycle': config.training.world_model_updates_per_cycle,
    'behavior_updates_per_cycle': config.training.behavior_updates_per_cycle,
    'policy_steps': config.training.policy_steps,
})

Run name: h100_screen_run_001
Saved config snapshot to: /content/drive/MyDrive/CSC_580_Final_Project/artifacts/training_runs/h100_screen_run_001/screening_config.yaml
{'actor_lr': 3e-05,
 'batch_size': 128,
 'behavior_updates_per_cycle': 4,
 'critic_lr': 5e-05,
 'imagination_horizon': 8,
 'max_episode_steps': 60,
 'policy_steps': 32,
 'world_model_lr': 0.0003,
 'world_model_updates_per_cycle': 24}


In [ ]:
EVAL_CYCLE_MARKS = [25, 50, 100, 150, 250, 400, 800]
DEMO_EPISODES_BY_STAGE = {25: 2, 50: 2, 100: 3, 150: 4, 250: 4, 400: 4, 800: 6}
MIN_AVG_REWARD_BY_STAGE = {25: -1.0, 50: 0.0, 100: 1.5, 150: 3.0, 250: 5.0, 400: 7.0, 800: 10.0}
MIN_AVG_STEPS_BY_STAGE = {25: 4.0, 50: 6.0, 100: 8.0, 150: 10.0, 250: 12.0, 400: 15.0, 800: 18.0}

STAGES = [
    {
        'label': f'screen_{cycles:04d}',
        'cycles': cycles,
        'demo_episodes': DEMO_EPISODES_BY_STAGE[cycles],
        'min_avg_reward': MIN_AVG_REWARD_BY_STAGE[cycles],
        'min_avg_steps': MIN_AVG_STEPS_BY_STAGE[cycles],
    }
    for cycles in EVAL_CYCLE_MARKS
]

AUTO_STOP = True
DEMO_MAX_STEPS = 200

for stage in STAGES:
    print(stage)

{'label': 'screen_150', 'cycles': 150, 'demo_episodes': 4, 'min_avg_reward': 3.0, 'min_avg_steps': 10.0}
{'label': 'screen_350', 'cycles': 350, 'demo_episodes': 4, 'min_avg_reward': 6.0, 'min_avg_steps': 14.0}
{'label': 'screen_800', 'cycles': 800, 'demo_episodes': 6, 'min_avg_reward': 10.0, 'min_avg_steps': 18.0}


In [ ]:
def summarize_demo(bundle):
    rewards = [result.total_reward for result in bundle.results]
    steps = [result.steps for result in bundle.results]
    crashes = [bool(result.terminated) for result in bundle.results]
    summary = {
        'real_eval/avg_reward': float(sum(rewards) / len(rewards)),
        'real_eval/avg_steps': float(sum(steps) / len(steps)),
        'real_eval/crash_rate': float(sum(crashes) / len(crashes)),
        'real_eval/min_reward': float(min(rewards)),
        'real_eval/max_reward': float(max(rewards)),
    }
    return summary


def stage_should_stop(stage, demo_summary):
    too_low_reward = demo_summary['real_eval/avg_reward'] < stage['min_avg_reward']
    too_short = demo_summary['real_eval/avg_steps'] < stage['min_avg_steps']
    return too_low_reward and too_short

In [ ]:
latest_checkpoint = None
stage_history = []
training_summary = None

for stage in STAGES:
    print(f"\n=== Stage: {stage['label']} | target_cycles={stage['cycles']} ===")
    training_summary = run_training_experiment(
        config,
        RUN_ARTIFACT_ROOT,
        cycles=stage['cycles'],
        warm_start_steps=config.training.warm_start_steps,
        policy_steps=config.training.policy_steps,
        checkpoint_interval=config.training.checkpoint_interval,
        resume_from=latest_checkpoint,
    )
    latest_checkpoint = training_summary.latest_checkpoint

    demo_outputs = record_demo_videos(
        config,
        checkpoint_path=latest_checkpoint,
        output_dir=RUN_ARTIFACT_ROOT / 'stage_demos' / stage['label'],
        num_episodes=stage['demo_episodes'],
        max_steps=DEMO_MAX_STEPS,
        fps=15,
        seed=config.seed,
        prefix=f"{RUN_NAME}_{stage['label']}",
        device=config.device,
    )

    demo_summary = summarize_demo(demo_outputs)
    latest_metrics = training_summary.latest_record
    stage_record = {
        'stage': stage['label'],
        'cycles': stage['cycles'],
        'checkpoint': str(latest_checkpoint),
        'world_model/total_loss': latest_metrics.get('world_model/total_loss'),
        'world_model/reward_loss': latest_metrics.get('world_model/reward_loss'),
        'behavior/actor_loss': latest_metrics.get('behavior/actor_loss'),
        'behavior/critic_loss': latest_metrics.get('behavior/critic_loss'),
        'behavior/imagined_reward_mean': latest_metrics.get('behavior/imagined_reward_mean'),
        'behavior/imagined_value_mean': latest_metrics.get('behavior/imagined_value_mean'),
        **demo_summary,
    }
    stage_history.append(stage_record)
    print(json.dumps(stage_record, indent=2))

    if AUTO_STOP and stage_should_stop(stage, demo_summary):
        print('Auto-stop triggered: real driving still looks too weak for this stage.')
        break

print('Latest checkpoint:', latest_checkpoint)


=== Stage: screen_150 | target_cycles=150 ===
[optimizer] cast models to bfloat16 for FlashAdamW
[optimizer] using FlashAdamW
[optimizer] using FlashAdamW
[optimizer] using FlashAdamW
[train] starting run | cycles=150 | start_step=1 | warm_start_steps=1000 | policy_steps=32 | device=cuda


In [ ]:
stage_history_path = RUN_ARTIFACT_ROOT / 'stage_history.json'
stage_history_path.write_text(json.dumps(stage_history, indent=2), encoding='utf-8')
print('Saved stage history to:', stage_history_path)
stage_history

## Optional: promote the best checkpoint to a longer run

If the stage results look good, rerun the training cell with a longer final stage or change the last stage target from `800` to `1200` or `1600`. This keeps expensive runs gated by real demo quality rather than imagination metrics alone.